### Binary Classification using Logistic Regression

* Dataset: Generate data for binary classification
* Objective: Minimize the binary cross-entropy function loss function 

$$
L(\theta) = - \frac{1}{n}\sum_{i=1}^{n} (y_i ln(p_i) + (1 - y_i)ln(1-p_i))
$$

Where
* n is the number of observations
* $y_i$ is the actual binary label (0, 1) of the $i^{th}$ obervation
* $p_i$ is the predicted probability of the $i^{it}$ observation being in class 1.

Sigmoid Function:

$$ \sigma (z) = \frac{1}{1+e^{-z}} ~~~~,~~ \text{Where}~~z=\theta^{T}x + b$$
The probability can be calculated using the stochastic function

$p_i (y=1|0;\theta) = \sigma(\theta^{T} x_i)$ , Where $\theta$ is the training parameters



## Task

* Train the model using CG and BFGS

## DataSet

https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

It is a binary classification dataset (malignant vs. benign tumor) based on digitized images of fine-needle aspirate (FNA) of breast mass.

The dataset includes information about:

It has 30 numeric features, all computed from images of cell nuclei. These features are grouped into three categories for each measurement type:

#### 1. Mean values
#### 2. Standard error (SE)
#### 3. Worst (largest mean) values

* radius – mean distance from center to points on perimeter

* texture – standard deviation of gray-scale values

* perimeter – tumor perimeter

* area – tumor area

* smoothness – local variation in radius lengths

* compactness – (perimeter² / area − 1.0)

* concavity – severity of concave portions of the contour

* concave points – number of concave portions of the contour

* symmetry – symmetry of the tumor shape

* fractal dimension – “coastline approximation” of the boundary

### Target

Binary classification:

* 0 → malignant  (No cancer)
* 1 → benign     (Have cancer)

### 

In [1]:
# Surpress warnings:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

In [2]:
import numpy as np 
from scipy.optimize import minimize
from math import log
import pandas as pd
from sklearn import preprocessing
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

#### Importing data

In [3]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

#Preprocessing data
X = preprocessing.StandardScaler().fit(X).transform(X)

# Train-test split 80% Data for trianing the model and 20% for test
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.2, random_state=4)


* Since the dataset have 30 features so the hypothesis function will be:

$$h_\theta = \sigma(\theta_0 + \theta_1 x_1[i] + \theta_2 x_2[i] + \theta_3 x_3[i] + .......... + \theta_{30} x_{30}[i])$$

In [4]:
def objective_fun(theta, x, y):
    z = x.dot(theta)
    z = np.clip(z, -500, 500)  # avoid overflow
    p = 1 / (1 + np.exp(-z))  # sigmoid
    eps = 1e-10               # avoid log(0)
    return -1/len(y) * np.sum(y*np.log(p+eps) + (1-y)*np.log(1-p+eps))

### Using Constrained Gradient (CG) Method to minimize the objective function

In [5]:
x0 = np.zeros(train_X.shape[1])   # initialize theta
result_CG = minimize(objective_fun, x0=x0, args=(train_X, train_y), method='CG')

print("Optimized thetas:\n", result_CG.x)


Optimized thetas:
 [  17.20510461  -22.65978965  -54.81326194  114.74315126   -7.06768487
   85.27291478  -49.85857799  -54.49469991   10.37474824  -24.25554738
 -117.96352049    8.15133473   62.93730755  -45.90950764    5.11004723
  -85.7794549     0.19987795  -30.98632289   46.44753198  120.65041152
 -107.59796273  -12.43406397  -93.70794383   81.94286583    3.38919898
   86.63477633  -19.30357995  -17.42302426  -43.21940453  -66.41789636]


#### Using BFGS Method

In [6]:
x0 = np.zeros(train_X.shape[1])
result_bfgs = minimize(objective_fun, x0=x0, args=(train_X, train_y), method='BFGS')

print('Optimized thetas:\n',result_bfgs.x)

Optimized thetas:
 [  152.31711637  -204.16063605   103.28630837   850.99913705
  -228.35636212  1201.36747096  -439.49591368 -1171.94075138
    75.26233654  -276.8114297  -1193.98494763   297.71177142
   541.28918697  -352.47673812   128.79238071 -1099.89877809
   -96.92952286  -561.35186619   550.75618707  1597.51267889
 -1208.62785855  -343.76701819  -355.06015393   -80.22692284
   172.46286317  1040.69698896  -276.71200629    34.56654867
  -421.10337295  -996.71597001]


### Prediction of CG

In [7]:
def predict(x, theta):
    z = x.dot(theta)
    z = np.clip(z, -500, 500)  # avoid overflow
    p = 1 / (1 + np.exp(-z))
    return (p >= 0.5).astype(int)   # classification threshold at 0.5

y_pred = predict(test_X, result_CG.x)


print('\nPredicted results:\n',y_pred)
print('\n\n Actual results:\n', test_y)

#Checking the accuracy
accuracy = np.mean(y_pred == test_y)
print("\n \n Training Accuracy: ", round(accuracy*100, 3),'%')



Predicted results:
 [0 1 0 0 1 0 1 1 0 0 0 1 1 1 1 1 1 1 1 1 1 1 0 1 0 0 1 1 1 1 1 1 1 1 1 0 1
 0 0 0 0 1 0 1 1 1 1 0 1 0 0 0 1 0 1 1 0 1 1 0 1 1 1 0 1 1 1 1 0 0 0 1 1 0
 1 0 0 1 1 1 1 0 0 1 1 1 1 1 0 1 1 0 1 0 0 1 1 1 1 1 1 0 0 1 1 1 1 1 1 0 1
 1 0 1]


 Actual results:
 [1 1 0 0 1 1 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 1 0 1
 0 0 0 0 1 0 1 1 1 1 0 1 0 0 1 1 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 0 0 1 1 1 0
 1 0 0 1 1 1 1 0 0 1 1 1 1 1 0 1 1 0 1 0 0 1 0 1 1 1 1 0 0 1 1 1 1 1 1 0 1
 0 0 1]

 
 Training Accuracy:  91.228 %


### Prediction of BFGS

In [10]:
def predict(x, theta):
    z = x.dot(theta)
    z = np.clip(z, -500, 500)  # avoid overflow
    p = 1 / (1 + np.exp(-z))
    return (p >= 0.5).astype(int)   # classification threshold at 0.5

y_pred = predict(test_X, result_bfgs.x)


print('\nPredicted results:\n',y_pred)
print('\n\n Actual results:\n', test_y)

#Checking the accuracy
accuracy = np.mean(y_pred == test_y)
print("\n \n Training Accuracy: ", round(accuracy*100, 3),'%')


Predicted results:
 [0 1 0 0 1 0 1 1 0 0 0 1 1 1 1 1 1 1 1 1 1 1 0 1 0 0 1 1 1 1 1 1 1 1 1 0 1
 0 0 0 0 1 0 1 1 1 1 0 1 0 0 1 1 0 1 1 0 1 1 0 1 1 1 0 1 1 1 1 0 0 0 1 1 0
 1 0 0 1 1 1 1 0 0 1 1 1 1 1 0 1 1 0 1 0 0 1 1 1 1 1 1 0 0 1 1 1 1 1 1 0 1
 1 1 1]


 Actual results:
 [1 1 0 0 1 1 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 1 0 1
 0 0 0 0 1 0 1 1 1 1 0 1 0 0 1 1 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 0 0 1 1 1 0
 1 0 0 1 1 1 1 0 0 1 1 1 1 1 0 1 1 0 1 0 0 1 0 1 1 1 1 0 0 1 1 1 1 1 1 0 1
 0 0 1]

 
 Training Accuracy:  91.228 %
